<a href="https://colab.research.google.com/github/JobaAdewumi/Fraud-SMS-Classifier/blob/main/Fraud_SMS_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import classification_report, confusion_matrix
import joblib

from utils import SMSTextCleaner

In [45]:
df = pd.read_csv('SMS_SPAM_DATA.csv')
df.head()

,Status,Spam,Message
0,receive,1,Dear Customer as a G-BAM customer you have r...
1,receive,1,Dear Customer as a G-BAM customer you have r...
2,receive,1,Want to win a Walker? Click here www.Facebook....
3,receive,1,Want to win a Walker? Click here www.Facebook....
4,receive,1,GREAT NEWS!We celebrate Nigeria's 100 years &w...


In [46]:
df.isnull().sum()

,0
Status,0
Spam,0
Message,0


In [47]:
df.drop(columns='Status', inplace=True)

In [48]:
df.head()

,Spam,Message
0,1,Dear Customer as a G-BAM customer you have r...
1,1,Dear Customer as a G-BAM customer you have r...
2,1,Want to win a Walker? Click here www.Facebook....
3,1,Want to win a Walker? Click here www.Facebook....
4,1,GREAT NEWS!We celebrate Nigeria's 100 years &w...


In [49]:
# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(df['Message'], df['Spam'], test_size=0.25, random_state=42)

In [50]:
# 3. Build and Train the Pipeline (Cleaner -> Vectorizer -> Classifier)
# Using Logistic Regression because the coefficients clearly show which words trigger the fraud flag
pipeline = Pipeline([
    ('cleaner', SMSTextCleaner()),
    ('tfidf', TfidfVectorizer(stop_words='english', lowercase=True)),
    ('clf', LogisticRegression(random_state=42))
])

In [51]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('cleaner', SMSTextCleaner()),
                ('tfidf', TfidfVectorizer(stop_words='english')),
                ('clf', LogisticRegression(random_state=42))])

In [52]:
# 4. Evaluate the Model (Fulfills the Metrics requirement)
y_pred = pipeline.predict(X_test)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.89      0.92      0.91       695
           1       0.89      0.86      0.88       551

    accuracy                           0.89      1246
   macro avg       0.89      0.89      0.89      1246
weighted avg       0.89      0.89      0.89      1246



In [53]:
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


--- Confusion Matrix ---
[[639  56]
 [ 75 476]]


In [54]:
# 5. Extract Global Keyword Insights (Feature Importance)
vectorizer = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['clf']

In [55]:
feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

In [56]:
importance_df = pd.DataFrame({
    'Keyword': feature_names,
    'Importance': coefficients
}).sort_values(by='Importance', ascending=False)

In [57]:
print("\n--- Top 5 Fraud-Indicating Keywords ---")
print(importance_df.head(5))


--- Top 5 Fraud-Indicating Keywords ---
     Keyword  Importance
6267    text    4.159112
3719    life    2.845725
4194     mtn    2.510740
5843     sms    2.388920
2068    ello    2.281992


In [58]:
# 6. Serialize and Save the Pipeline
joblib.dump(pipeline, 'sms_fraud_model.joblib')
print("\nModel successfully saved as 'sms_fraud_model.joblib'")


Model successfully saved as 'sms_fraud_model.joblib'
